# Notebook da Camada Bronze

Notebook (2/5) da Tarefa 2 — Projeto CineData Analytics

Este notebook faz a ingestão dos dados brutos. Ele lê os cinco CSVs do volume `landing` e grava uma tabela Delta para cada um no schema `bronze`. Também extrai a cotação do dólar (PTAX) da API do Banco Central para o período informado nos widgets do notebook (por padrão, os últimos 7 dias) e grava em `bronze.tb_cotacao_dolar`.

## Decisões adotadas
- **Dado cru, sem alteração:** todas as colunas são lidas como texto, sem `inferSchema`. Valores como `"Unknown"` no meio de colunas numéricas são preservados como vieram, e a tipagem e a limpeza ficam para a Silver.
- **Leitura robusta de CSV:** as opções `multiLine` e `escape` tratam textos longos (sinopses, comentários) com vírgulas, aspas e quebras de linha internas, que de outra forma quebrariam as linhas.
- **Rastreabilidade:** cada tabela recebe a coluna `ingestion_datetime`, com o momento exato da carga.
- **Modo `append`:** cada execução acrescenta registros sem apagar as cargas anteriores. Por isso, reexecuções geram duplicatas por design. A deduplicação acontece na Silver, que usa o `ingestion_datetime` para manter a versão mais recente.
- **Padrão de nome de arquivo:** a leitura usa curinga (`*movies_info*.csv`) porque os nomes dos arquivos têm prefixos numéricos e sufixos inconsistentes (`TMDB_IMDB` / `IMDB_TMDB`).
- **API do Banco Central:**
  - As datas vêm de widgets no formato `MM-DD-AAAA`, exigido pela API, e são validadas antes da chamada.
  - O padrão de 7 dias corridos garante que haja dias úteis no período, já que a API não retorna cotação em fins de semana e feriados.
  - `raise_for_status()` interrompe a execução se a API responder com erro HTTP.
  - Se o período não tiver cotações, nada é gravado e o notebook apenas avisa.

In [0]:
# Esta celula serve para definição dos parametros de data do notebook (usado na extração da API)
# Eles podem ser modificados diretamente no cabeçalho do notebook
from datetime import date, timedelta

# definindo as datas de hoje e de 7 dias atrás
hoje = date.today()
padrao_inicio = (hoje - timedelta(days=7)).strftime("%m-%d-%Y")
padrao_fim = hoje.strftime("%m-%d-%Y")

# definindo os widgets
dbutils.widgets.text("data_inicio", padrao_inicio, "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", padrao_fim, "Data fim (MM-DD-AAAA)")

## Criação das tabelas no schema Bronze
Esta célula roda um for, de tabela a tabela carregando os dados contidos nos csvs

In [0]:
from pyspark.sql import functions as F

catalog = "cinedata_medallion"
bronze_schema = f"{catalog}.bronze"
volume_path = f"/Volumes/{catalog}/bronze/landing"


# Tabelas Bronze a partir dos CSVs
tabelas = ["movies_info", "movies_financials", "movies_metrics",
           "credits_and_tags", "movies_reviews"]

# Loop de criação das tabelas
for tabela in tabelas:
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)
          .option("quote", '"')
          .option("escape", '"')
          .csv(f"{volume_path}/*{tabela}*.csv")
          .withColumn("ingestion_datetime", F.current_timestamp()))


    (df.write     # escreve a tabela como delta em modo append
       .format("delta")
       .mode("append")
       .saveAsTable(f"{bronze_schema}.{tabela}"))

    print(f"{bronze_schema}.{tabela}: {df.count()} linhas lidas")

## Extração e Criação da tabela de Cotação
Esta célula faz a requisição para a API do Banco Central com as cotações, armazena em um df se obtiver resposta e carrega na tabela designada

In [0]:
# TRUNCATE TABLE caso desejado
#spark.sql(f"TRUNCATE TABLE {bronze_schema}.tb_cotacao_dolar")
#print(f"Tabela {bronze_schema}.tb_cotacao_dolar truncada com sucesso.")

In [0]:
import requests
from datetime import datetime
from pyspark.sql import functions as F

# destino na bronze
catalog = "cinedata_medallion"
tabela_cotacao = f"{catalog}.bronze.tb_cotacao_dolar"

# datas vindas dos widgets (formato MM-DD-AAAA)
data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

# valida o formato antes de chamar a API (erro aqui = data digitada errada)
for d in (data_inicio_formatada, data_fim_formatada):
    datetime.strptime(d, "%m-%d-%Y")

# endpoint PTAX do Banco Central: cotações do período, só data/hora e valor de compra
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_formatada}'"
    f"&@dataFinalCotacao='{data_fim_formatada}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

# chamada à API; para a execução se a resposta for erro HTTP
resposta = requests.get(url, timeout=30)
resposta.raise_for_status()
cotacoes = resposta.json()["value"]

print(f"Período: {data_inicio_formatada} a {data_fim_formatada} | {len(cotacoes)} cotações retornadas")

# grava só se houver cotações (período só com fim de semana/feriado volta vazio)
if cotacoes:
    df = (spark.createDataFrame(cotacoes)
          .select("dataHoraCotacao", "cotacaoCompra")
          .withColumn("ingestion_datetime", F.current_timestamp()))  # timestamp da carga

    # bronze em append: cada execução acrescenta, sem apagar cargas anteriores
    (df.write
       .format("delta")
       .mode("append")
       .saveAsTable(tabela_cotacao))

    display(spark.table(tabela_cotacao).orderBy(F.desc("ingestion_datetime")))
else:
    print("Nenhuma cotação no período (fim de semana/feriado?). Nada gravado.")

In [0]:
# Celula adicional mostrando todas as tabelas da bronze carregadas (serviu de apoio para as transformações feitas na camada silver)

for tabela in tabelas:
    display(spark.table(f"{bronze_schema}.{tabela}").orderBy(F.desc("id")))                   